# 2.4. Verificación de la Calidad de Datos

Fase 2.4 de CRISP-DM: evaluación algorítmica de completitud, consistencia, valores fuera de rango y ruido en el dataset.

In [ ]:
import json
import pandas as pd
import numpy as np

DATA_DIR = "../data"
CSV_PATH = f"{DATA_DIR}/GBvideos_cc50_202101.csv"
JSON_PATH = f"{DATA_DIR}/GB_category_id.json"

In [ ]:
# Carga y mapeo de categorías
df = pd.read_csv(CSV_PATH)
with open(JSON_PATH, "r", encoding="utf-8") as f:
    categories_data = json.load(f)
category_map = {
    int(item["id"]): item["snippet"]["title"] for item in categories_data["items"]
}
df["category_name"] = df["category_id"].map(category_map).fillna("Unknown")

# Conversión de tipos
df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce")
df["trending_date"] = pd.to_datetime(df["trending_date"], errors="coerce")
print("Shape:", df.shape)

## 2.4.1 Valores nulos

In [ ]:
missing = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
display(pd.concat([missing, missing_pct], axis=1, keys=['nulos','pct_nulos']))

In [ ]:
# Visualización de nulos (requiere missingno)
try:
    import missingno as msno
    msno.matrix(df)
except ImportError:
    print('missingno no está instalado')

## 2.4.2 Duplicados

In [ ]:
# Duplicados exactos, por video_id y por (video_id, trending_date)
n0 = len(df)
dup_exactos = df.duplicated().sum()
dup_video = df.duplicated(subset=["video_id"]).sum()
dup_video_fecha = df.duplicated(subset=["video_id", "trending_date"]).sum()
print(f"Duplicados exactos: {dup_exactos}")
print(f"Duplicados por video_id: {dup_video}")
print(f"Duplicados por (video_id, trending_date): {dup_video_fecha}")

# Conservamos la última aparición en trending de cada video
df_clean = (
    df.sort_values("trending_date")
    .drop_duplicates(subset=["video_id"], keep="last")
    .reset_index(drop=True)
)
print(f"Filas antes: {n0} | después: {len(df_clean)}")

## 2.4.3 Inconsistencias lógicas

In [ ]:
# Flags de inconsistencias lógicas
df_clean["likes_gt_views"] = df_clean["likes"] > df_clean["views"]
df_clean["dislikes_gt_views"] = df_clean["dislikes"] > df_clean["views"]
df_clean["comments_gt_views"] = df_clean["comment_count"] > df_clean["views"]
df_clean["negative_likes"] = df_clean["likes"] < 0
df_clean["negative_dislikes"] = df_clean["dislikes"] < 0
df_clean["invalid_geo"] = (
    df_clean["lat"].isna()
    | df_clean["lon"].isna()
    | (df_clean["lat"] < 49)
    | (df_clean["lat"] > 61)
    | (df_clean["lon"] < -9)
    | (df_clean["lon"] > 2)
)
df_clean["unknown_category"] = df_clean["category_name"] == "Unknown"

error_cols = [
    "likes_gt_views", "dislikes_gt_views", "comments_gt_views",
    "negative_likes", "negative_dislikes", "invalid_geo", "unknown_category",
]
print(df_clean[error_cols].sum())

## 2.4.4 Outliers

In [ ]:
# Límites de outliers por IQR
metricas = ["views", "likes", "dislikes", "comment_count"]
for col in metricas:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    n_out = ((df_clean[col] < lim_inf) | (df_clean[col] > lim_sup)).sum()
    pct = round(n_out / len(df_clean) * 100, 2)
    print(f"{col}: [{lim_inf:.1f}, {lim_sup:.1f}] | outliers = {n_out} ({pct}%)")

In [ ]:
# Detección de outliers por Z-score (umbral 3)
for col in metricas:
    mean = df_clean[col].mean()
    std = df_clean[col].std()
    z = (df_clean[col] - mean).abs() / std
    df_clean[f"{col}_outlier_z"] = z > 3
    print(f"{col}: outliers Z>3 = {df_clean[f'{col}_outlier_z'].sum()}")

print("\nNota: los valores extremos corresponden a viralidad orgánica (ley de Pareto),")
print("no a errores técnicos, por lo que se conservan para el análisis.")

## 2.4.5 Reporte de calidad

In [ ]:
# Reporte de calidad
print("Shape:", df_clean.shape)
print("Memoria (MB):", round(df_clean.memory_usage(deep=True).sum() / 1024**2, 2))
print("Video_id duplicados:", df_clean["video_id"].duplicated().sum())
print("\nNulos por columna (>0):")
nulos = df_clean.isna().sum()
print(nulos[nulos > 0])

## 2.4.6 Limpieza final

In [ ]:
# Limpieza final: eliminamos registros con error, categoría desconocida o geografía inválida
mask = (
    ~df_clean["video_error_or_removed"].astype(bool)
    & ~df_clean["unknown_category"]
    & ~df_clean["invalid_geo"]
    & ~df_clean["likes_gt_views"]
    & ~df_clean["dislikes_gt_views"]
)
df_final = df_clean[mask].reset_index(drop=True)
print("Shape final:", df_final.shape)